# Covariate-scaled multinomial logit

## Model

The utility scale can vary with observed characteristics:

$$
s_{nj}=\exp(\boldsymbol z_n^\mathsf T\boldsymbol\gamma_j),\qquad
P_{nj}=
\frac{\exp(V_{nj}/s_{nj})}
{\sum_{k\in\mathcal{C}_n}\exp(V_{nk}/s_{nk})}.
$$

This example uses standardized income to explain the TRAIN and CAR scales,
while the SM scale is fixed to one. Utility and scale parameters are estimated
jointly.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import torch

from IPython.display import HTML, display

from torchdcm import (
    Beta,
    ChoiceDataset,
    CovariateScale,
    CovariateScaledMultinomialLogit,
    UtilitySpec,
)
from torchdcm.datasets import make_swissmetro_like

# Use double precision and a fixed seed for stable, reproducible estimation.
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
# Use CUDA automatically when available; set device = "cpu" to force CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
# Helper: map wide attributes and availability into a ChoiceDataset.
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


# Helper: define generic time and cost effects with SM as reference.
def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec

# Standardize income before using it in the log-scale equations.
frame = make_swissmetro_like(n_obs=180, seed=17)
frame["income_std"] = (
    (frame["income"] - frame["income"].mean()) / frame["income"].std()
)
# Include standardized income with the alternative-level attributes.
data = ChoiceDataset.from_wide(
    frame,
    alternatives=["TRAIN", "SM", "CAR"],
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
        "income_std": {
            "TRAIN": "income_std",
            "SM": "income_std",
            "CAR": "income_std",
        },
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)
spec = make_utility_spec()


In [3]:
# Let income shift TRAIN and CAR log-scales while SM remains fixed.
scales = {
    "TRAIN": CovariateScale(
        log_scale=Beta("GAMMA_TRAIN", init=0.05) * "income_std"
    ),
    "SM": CovariateScale(value=1.0),
    "CAR": CovariateScale(
        log_scale=Beta("GAMMA_CAR", init=-0.05) * "income_std"
    ),
}
model = CovariateScaledMultinomialLogit(
    spec,
    scales,
    device=device,
    max_iter=60,
)
result = model.fit(data)
# Fit utility and scale equations jointly, then render the report.
display(HTML(result.report().to_html()))


Model family,CovariateScaledMultinomialLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,60
Line search,strong_wolfe
